## 1. Data Partitioning and On-the-Fly Generation

To train the segmentation model, we first partition our dataset into Training (80%) and Validation (20%) sets using a fixed random seed to ensure reproducibility. 

Because medical imaging datasets are highly memory-intensive, we implement a custom tf.keras.utils.Sequence generator. This generator performs several crucial preprocessing steps on-the-fly:
1. **Hounsfield Unit (HU) Windowing:** Clips voxel densities to the [-30.0, 150.0] range to isolate skeletal muscle tissue.
2. **Min-Max Normalization:** Scales the HU values to a [0, 1] range for neural network stability.
3. **Spatial Resampling:** Resizes images to 256x256 using bilinear interpolation for the scans and nearest-neighbor interpolation for the masks (to preserve the discrete 0, 1, 2 categorical labels).

In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import math
import numpy as np
import nibabel as nib
import tensorflow as tf

# 1. Recuperiamo tutti i file
DIR_IMMAGINI = Path(r"C:\Users\vella\Desktop\tirocinio\Dataset\slices")
DIR_MASCHERE = Path(r"C:\Users\vella\Desktop\tirocinio\Dataset\masks")

tutte_le_immagini = sorted(list(DIR_IMMAGINI.glob("*.nii")))
tutte_le_maschere = sorted(list(DIR_MASCHERE.glob("*.nii")))

# 2. Dividiamo i dati: 80% Training, 20% Validation
train_img, val_img, train_mask, val_mask = train_test_split(
    tutte_le_immagini, tutte_le_maschere, test_size=0.2, random_state=42
)

print(f"Training patients: {len(train_img)}")
print(f"Validation patients: {len(val_img)}")

# 3. Aggiorniamo LEGGERMENTE il generatore per accettare le liste di file invece delle cartelle
class L3MuscleDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, lista_immagini, lista_maschere, batch_size=4, target_size=(256, 256)):
        self.lista_immagini = lista_immagini
        self.lista_maschere = lista_maschere
        self.batch_size = batch_size
        self.target_size = target_size

    def __len__(self):
        return math.ceil(len(self.lista_immagini) / self.batch_size)

    def __getitem__(self, idx):
        batch_x_paths = self.lista_immagini[idx * self.batch_size : (idx + 1) * self.batch_size]
        batch_y_paths = self.lista_maschere[idx * self.batch_size : (idx + 1) * self.batch_size]

        batch_x, batch_y = [], []

        for img_path, mask_path in zip(batch_x_paths, batch_y_paths):
            img_np = np.squeeze(nib.load(img_path).get_fdata())
            mask_np = np.squeeze(nib.load(mask_path).get_fdata())

            HU_MIN, HU_MAX = -30.0, 150.0
            img_np = np.clip(img_np, HU_MIN, HU_MAX)
            img_np = (img_np - HU_MIN) / (HU_MAX - HU_MIN) 

            img_tensor = tf.expand_dims(tf.convert_to_tensor(img_np, dtype=tf.float32), axis=-1) 
            mask_tensor = tf.expand_dims(tf.convert_to_tensor(mask_np, dtype=tf.int32), axis=-1)

            img_resized = tf.image.resize(img_tensor, self.target_size, method='bilinear')
            mask_resized = tf.image.resize(mask_tensor, self.target_size, method='nearest')

            batch_x.append(img_resized.numpy())
            batch_y.append(mask_resized.numpy())

        return np.array(batch_x), np.array(batch_y)

# 4. Creiamo DUE generatori
train_generator = L3MuscleDataGenerator(train_img, train_mask, batch_size=4)
val_generator = L3MuscleDataGenerator(val_img, val_mask, batch_size=4)

## 2. Custom Objective Functions: Overcoming Class Imbalance

In abdominal CT scans, muscle tissue occupies a very small percentage of the total image area compared to the background. Relying solely on standard accuracy or Cross-Entropy loss would lead to a biased model that over-predicts the background class.

To mitigate this, we define a custom **Combined Loss Function**:
* **Sparse Categorical Crossentropy (SCCE):** Ensures pixel-level classification stability.
* **Dice Loss:** Evaluates the spatial overlap between the predicted mask and the ground truth, forcing the model to focus on the morphological shape of the Abdominals and Psoas muscles, ignoring the background entirely.

In [ ]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.losses import SparseCategoricalCrossentropy

def muscle_dice_coef(y_true, y_pred, smooth=1e-6):
    """
    Computes the Dice Coefficient for the Abdominals and Psoas classes,
    ignoring the background.
    """
    # 1. Squeeze the channel dimension if present and convert to one-hot
    y_true = tf.squeeze(y_true, axis=-1) 
    y_true = tf.cast(y_true, tf.int32)
    y_true_one_hot = tf.one_hot(y_true, depth=3) # depth = num_classes
    y_true_one_hot = tf.cast(y_true_one_hot, tf.float32)
    
    # 2. Slice to exclude the Background (class 0)
    # y_true_one_hot & y_pred shapes become: (batch, height, width, 2)
    y_true_target = y_true_one_hot[..., 1:] 
    y_pred_target = y_pred[..., 1:]
    
    # 3. Calculate intersection and denominator over spatial dimensions (H, W)
    axes = [1, 2] 
    intersection = K.sum(y_true_target * y_pred_target, axis=axes)
    denominator = K.sum(y_true_target, axis=axes) + K.sum(y_pred_target, axis=axes)
    
    # 4. Calculate Dice per class, then average across classes and the batch
    dice_per_class = (2. * intersection + smooth) / (denominator + smooth)
    return K.mean(dice_per_class)

def muscle_dice_loss(y_true, y_pred):
    return 1.0 - muscle_dice_coef(y_true, y_pred)

def combined_loss(y_true, y_pred):
    """
    Combines Cross-Entropy for pixel-level stability with Dice for shape accuracy.
    """
    scce_loss = SparseCategoricalCrossentropy()(y_true, y_pred)
    dice_loss = muscle_dice_loss(y_true, y_pred)
    return scce_loss + dice_loss

## 3. Physical Boundary Evaluation Metrics

While the Dice Coefficient measures volumetric overlap, clinical applications require precise boundary delineation. We implement two boundary-distance metrics calculated dynamically in **millimeters (mm)** rather than pixels, utilizing the native spatial resolution stored in the NIfTI headers:

1. **Maximum Hausdorff Distance (Max HD):** Captures the absolute worst-case boundary deviation, identifying severe morphological failures.
2. **Balanced Average Hausdorff Distance (bAHD):** Measures the mean average error between the predicted contour and the ground truth contour.


In [ ]:
import numpy as np
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree

def get_boundary_coordinates(mask):
    """
    Extracts the pixel coordinates of the boundary of a binary mask.
    Uses morphological erosion: Boundary = Original - Eroded.
    (Rimane invariata perché lavora sulla griglia di pixel nativa)
    """
    # Erode the mask by 1 pixel
    eroded_mask = binary_erosion(mask)
    
    # XOR operation isolates the 1-pixel thick boundary
    boundary = mask ^ eroded_mask 
    
    # Return the (y, x) coordinates of the boundary pixels
    return np.argwhere(boundary)

In [ ]:
def calculate_HD_single_class_mm(y_true_binary, y_pred_binary, spacing_y_mm, spacing_x_mm):
    """
    Calculates the Standard (Maximum) Hausdorff Distance for a single binary class directly in millimeters.
    Represents the absolute worst-case boundary error in the physical space.
    """
    # 1. Extract boundary coordinate sets X and Y
    X = get_boundary_coordinates(y_true_binary)
    Y = get_boundary_coordinates(y_pred_binary)
    
    # Edge Cases: 
    if len(X) == 0 and len(Y) == 0:
        return 0.0
    if len(X) == 0 or len(Y) == 0:
        return np.nan 
        
    # --- LA MODIFICA CRUCIALE: Convertiamo le coordinate da pixel a millimetri ---
    # Moltiplichiamo ogni coordinata [y, x] per la rispettiva dimensione del pixel
    X_mm = X * np.array([spacing_y_mm, spacing_x_mm])
    Y_mm = Y * np.array([spacing_y_mm, spacing_x_mm])
    
    # 2. Build KD-Trees for fast nearest-neighbor search usando i punti in millimetri
    tree_X = cKDTree(X_mm)
    tree_Y = cKDTree(Y_mm)
    
    # 3. Query the trees to find the minimum distances
    # dist_X_to_Y contains the shortest distance from every point in X to Y
    dist_X_to_Y, _ = tree_Y.query(X_mm)
    max_dist_X = np.max(dist_X_to_Y) # The absolute worst error starting from X
    
    # dist_Y_to_X contains the shortest distance from every point in Y to X
    dist_Y_to_X, _ = tree_X.query(Y_mm)
    max_dist_Y = np.max(dist_Y_to_X) # The absolute worst error starting from Y
    
    # 4. Standard Hausdorff is the maximum of these two extreme errors
    hd_mm = max(max_dist_X, max_dist_Y)
    
    return hd_mm

In [ ]:
def calculate_bAHD_single_class_mm(y_true_binary, y_pred_binary, spacing_y_mm, spacing_x_mm):
    """
    Calculates the Balanced Average Hausdorff Distance for a single binary class directly in millimeters.
    """
    # 1. Extract boundary coordinate sets X and Y (in pixel)
    X = get_boundary_coordinates(y_true_binary)
    Y = get_boundary_coordinates(y_pred_binary)
    
    # Edge Cases: 
    if len(X) == 0 and len(Y) == 0:
        return 0.0
    if len(X) == 0 or len(Y) == 0:
        return np.nan 
        
    # --- LA MODIFICA IN MILLIMETRI ---
    # Moltiplichiamo le coordinate dei pixel [y, x] per la loro dimensione fisica reale in mm
    X_mm = X * np.array([spacing_y_mm, spacing_x_mm])
    Y_mm = Y * np.array([spacing_y_mm, spacing_x_mm])
    
    # 2. Build KD-Trees usando i punti convertiti in millimetri
    tree_X = cKDTree(X_mm)
    tree_Y = cKDTree(Y_mm)
    
    # 3. Query the trees to find the minimum physical distances
    # dist_X_to_Y: per ogni punto reale in X, trova la distanza in mm dal punto più vicino in Y
    dist_X_to_Y, _ = tree_Y.query(X_mm)
    sum_dist_X = np.sum(dist_X_to_Y)
    
    # dist_Y_to_X: per ogni punto reale in Y, trova la distanza in mm dal punto più vicino in X
    dist_Y_to_X, _ = tree_X.query(Y_mm)
    sum_dist_Y = np.sum(dist_Y_to_X)
    
    # 4. Applichiamo la formula bAHD (il risultato ora è una distanza media reale in mm)
    bahd_mm = (sum_dist_X + sum_dist_Y) / (len(X) + len(Y))
    
    return bahd_mm

## 4. Custom U-Net Architecture and Training

We implement a deep, fully convolutional U-Net architecture. Due to the complex textures of striated muscle tissue in CT scans, the filter depth has been augmented (up to 256 filters in the bottleneck) to enhance feature extraction capabilities. 

The network is trained over 50 epochs, utilizing an Adam optimizer and a ModelCheckpoint callback designed to automatically monitor the validation combined loss and preserve only the optimal weights.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, concatenate, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint

# Impostazioni base
img_size = (256, 256)
num_classes = 3 # 0: Sfondo, 1: Addominali, 2: Psoas

# AUMENTATI i filtri per catturare le texture muscolari mediche (16, 32, 64, 128, 256)
filters = [16, 32, 64, 128, 256] 
i = 0

# 1. INPUT: Le CT scan hanno 1 solo canale, non 3 (non sono RGB)
inputs = Input(shape=img_size + (1,))

# Nessun Lambda(x/255) qui, perché i dati CT devono essere normalizzati con le Unità Hounsfield prima!
s = inputs 

# --- INIZIO ENCODER ---
c1 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(s)
c1 = BatchNormalization()(c1)
c1 = Dropout(0.1)(c1)
c1 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c1)
p1 = MaxPooling2D((2, 2))(c1)
i += 1

c2 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p1)
c2 = Dropout(0.1)(c2)
c2 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c2)
p2 = MaxPooling2D((2, 2))(c2)
i += 1

c3 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p2)
c3 = Dropout(0.2)(c3)
c3 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c3)
p3 = MaxPooling2D((2, 2))(c3)
i += 1

c4 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p3)
c4 = Dropout(0.2)(c4)
c4 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c4)
p4 = MaxPooling2D(pool_size=(2, 2))(c4)
i += 1

# --- BOTTLENECK ---
c5 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(p4)
c5 = Dropout(0.3)(c5)
c5 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c5)
i -= 1

# --- INIZIO DECODER ---
u6 = Conv2DTranspose(filters[i], (2, 2), strides=(2,2), padding='same')(c5)
u6 = concatenate([u6, c4])
c6 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u6)
c6 = Dropout(0.2)(c6)
c6 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c6)
i -= 1

u7 = Conv2DTranspose(filters[i], (2, 2), strides=(2, 2), padding='same')(c6)
u7 = concatenate([u7, c3])
c7 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u7)
c7 = Dropout(0.2)(c7)
c7 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c7)
i -= 1

u8 = Conv2DTranspose(filters[i], (2, 2), strides=(2, 2), padding='same')(c7)
u8 = concatenate([u8, c2])
c8 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u8)
c8 = Dropout(0.1)(c8)
c8 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c8)
i -= 1

u9 = Conv2DTranspose(filters[i], (2, 2), strides=(2, 2), padding='same')(c8)
u9 = concatenate([u9, c1], axis=3)
c9 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(u9)
c9 = Dropout(0.1)(c9)
c9 = Conv2D(filters[i], (3, 3), activation='relu', kernel_initializer='he_normal', padding='same')(c9)

# --- OUTPUT LAYER ---
# Softmax è perfetto per la multiclasse
outputs = Conv2D(num_classes, 3, padding='same', activation='softmax')(c9)

model = Model(inputs=[inputs], outputs=[outputs])

# compilation
# compilation
lr = 0.001
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr),
    loss=combined_loss,           # Replaced simple SCCE with Combined Loss
    metrics=[muscle_dice_coef, 'accuracy']  # Replaced 'accuracy' with the custom Dice
)

# save best model (Now monitoring the combined loss)
filepath = "unet_sarcopenia_best.keras" # Note: .keras is the modern standard over .h5
checkpoint = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

model.summary()

In [ ]:
print("Inizio dell'addestramento del modello...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=50,
    callbacks=callbacks_list,
    verbose=1
)

## 5. Post-Training Validation in Physical Space (Millimeters)

Once training concludes, we reload the best-performing weights from the disk. We then iterate over the entire validation set to calculate the final structural metrics. Crucially, the model dynamically accesses the metadata in the original NIfTI files to compute exact physical errors in millimeters, providing clinically relevant validation data.

In [ ]:
# --- VALUTAZIONE POST-TRAINING CON bAHD E MAX HD (IN MILLIMETRI) ---
print("\nCaricamento dei pesi migliori per la valutazione...")
model.load_weights(filepath)

abdominals_bahd_scores = []
psoas_bahd_scores = []
abdominals_hd_scores = []
psoas_hd_scores = []

print("Calcolo delle metriche di distanza in MILLIMETRI sul Validation Set (potrebbe richiedere qualche minuto)...")

# Impostazione fissa che hai usato nel generatore
TARGET_SIZE = 256 
batch_size = val_generator.batch_size

for i in range(len(val_generator)):
    X_batch, y_batch = val_generator[i]
    
    preds = model.predict(X_batch, verbose=0)
    pred_classes = np.argmax(preds, axis=-1)
    true_classes = np.squeeze(y_batch, axis=-1) 

    # Estraiamo i percorsi reali dei file NIfTI per questo specifico batch
    # Usiamo la lista 'val_img' creata in precedenza con il train_test_split
    batch_paths = val_img[i * batch_size : (i + 1) * batch_size]

    for j in range(X_batch.shape[0]):
        true_mask = true_classes[j]
        pred_mask = pred_classes[j]
        
        # --- 1. ESTRAZIONE DELLA SPAZIATURA IN MILLIMETRI DAL FILE ORIGINALE ---
        nii_file = nib.load(batch_paths[j])
        header = nii_file.header
        
        # .get_zooms() restituisce le dimensioni fisiche del pixel originale: (dx, dy, dz) in mm
        original_zooms = header.get_zooms()
        original_shape = nii_file.shape
        
        # Calcoliamo di quanto abbiamo ridimensionato l'immagine per passarla alla U-Net
        factor_x = original_shape[0] / TARGET_SIZE
        factor_y = original_shape[1] / TARGET_SIZE
        
        # Calcoliamo la nuova dimensione del pixel nell'immagine 256x256
        spacing_x_mm = original_zooms[0] * factor_x
        spacing_y_mm = original_zooms[1] * factor_y
        
        # --- 2. Evaluate Addominali (Class 1) ---
        true_abs = (true_mask == 1)
        pred_abs = (pred_mask == 1)
        
        # Passiamo le dimensioni in mm alle nuove funzioni!
        bahd_abs = calculate_bAHD_single_class_mm(true_abs, pred_abs, spacing_y_mm, spacing_x_mm)
        hd_abs = calculate_HD_single_class_mm(true_abs, pred_abs, spacing_y_mm, spacing_x_mm)
        
        if not np.isnan(bahd_abs): abdominals_bahd_scores.append(bahd_abs)
        if not np.isnan(hd_abs): abdominals_hd_scores.append(hd_abs)
            
        # --- 3. Evaluate Psoas (Class 2) ---
        true_psoas = (true_mask == 2)
        pred_psoas = (pred_mask == 2)
        
        bahd_psoas = calculate_bAHD_single_class_mm(true_psoas, pred_psoas, spacing_y_mm, spacing_x_mm)
        hd_psoas = calculate_HD_single_class_mm(true_psoas, pred_psoas, spacing_y_mm, spacing_x_mm)
        
        if not np.isnan(bahd_psoas): psoas_bahd_scores.append(bahd_psoas)
        if not np.isnan(hd_psoas): psoas_hd_scores.append(hd_psoas)

print("\n--- RISULTATI FINALI SUL VALIDATION SET (IN MILLIMETRI) ---")
print(f"ADDOMINALI -> Mean bAHD: {np.mean(abdominals_bahd_scores):.2f} mm | Max HD: {np.mean(abdominals_hd_scores):.2f} mm")
print(f"PSOAS      -> Mean bAHD: {np.mean(psoas_bahd_scores):.2f} mm | Max HD: {np.mean(psoas_hd_scores):.2f} mm")

## 6. Statistical Distribution of Boundary Errors

After computing the spatial metrics in millimeters, it is essential to visualize their statistical distribution across the entire validation cohort. 
We utilize Boxplots to represent the spread, median, and outliers of our two primary metrics:
* **Balanced Average Hausdorff Distance (bAHD):** Indicates the general contour alignment.
* **Maximum Hausdorff Distance (Max HD):** Highlights the worst-case boundary deviations.

By plotting these distributions side-by-side for both the Abdominals and Psoas classes, we can clinically validate the stability and reliability of the U-Net predictions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_step1_metrics(abdominals_bahd, psoas_bahd, abdominals_hd, psoas_hd):
    print("\nGenerazione del Grafico 1: Boxplot delle Metriche...")
    
    # Imposta un tema accademico pulito (sfondo bianco, niente griglie pesanti)
    sns.set_theme(style="ticks")
    
    # Crea la figura (1 riga, 2 colonne)
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=300)
    fig.patch.set_facecolor('white') # Previene problemi di visualizzazione con il Dark Mode
    
    # 1. Prepara i dati per il bAHD
    df_bahd = pd.DataFrame({
        'Addominali': abdominals_bahd,
        'Psoas': psoas_bahd
    })
    
    # Plotta il bAHD (Colore: Toni di Blu)
    sns.boxplot(data=df_bahd, ax=axes[0], width=0.4, palette="Blues", fliersize=4)
    axes[0].set_title('Balanced Average Hausdorff (bAHD)', fontsize=12, fontweight='bold', pad=10)
    axes[0].set_ylabel('Errore Medio (Millimeters)', fontsize=11)
    axes[0].set_ylim(bottom=0) # Forza l'asse Y a partire da 0
    
    # 2. Prepara i dati per il Max HD
    df_hd = pd.DataFrame({
        'Addominali': abdominals_hd,
        'Psoas': psoas_hd
    })
    
    # Plotta il Max HD (Colore: Toni di Rosso)
    sns.boxplot(data=df_hd, ax=axes[1], width=0.4, palette="Reds", fliersize=4)
    axes[1].set_title('Maximum Hausdorff (Worst-Case)', fontsize=12, fontweight='bold', pad=10)
    axes[1].set_ylabel('Errore Massimo (Millimeters)', fontsize=11)
    axes[1].set_ylim(bottom=0) # Forza l'asse Y a partire da 0
    
    # Rimuovi i bordi in alto e a destra per un look più pulito
    sns.despine()
    
    # Salva e mostra
    plt.tight_layout()
    plt.savefig("thesis_Step1_Metrics.png", bbox_inches='tight', facecolor='white')
    plt.show()

# --- ESEGUI LA FUNZIONE ---
plot_step1_metrics(
    abdominals_bahd_scores, 
    psoas_bahd_scores, 
    abdominals_hd_scores, 
    psoas_hd_scores
)

## 7. Qualitative Visual Inspection Framework

Quantitative metrics must always be accompanied by qualitative visual inspections to ensure the model's anatomical logic aligns with human medical expertise. 

To achieve a clean visualization, we define a dynamic bounding box algorithm (crop_to_body). This function automatically trims the vast areas of background air (HU = -1000, mapped to near 0) surrounding the patient on the CT scan. This isolates the anatomical region of interest, allowing for a high-resolution side-by-side comparison of the Original CT, the Ground Truth mask, and the AI Prediction.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import numpy as np

def crop_to_body(ct_slice, mask_true, mask_pred, margin=30):
    body_pixels = ct_slice > 0.05 
    rows = np.any(body_pixels, axis=1)
    cols = np.any(body_pixels, axis=0)
    
    if not np.any(rows) or not np.any(cols):
        return ct_slice, mask_true, mask_pred 
        
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    
    rmin, rmax = max(0, rmin-margin), min(ct_slice.shape[0], rmax+margin)
    cmin, cmax = max(0, cmin-margin), min(ct_slice.shape[1], cmax+margin)
    
    return ct_slice[rmin:rmax, cmin:cmax], mask_true[rmin:rmax, cmin:cmax], mask_pred[rmin:rmax, cmin:cmax]

def plot_step2_segmentation(ct_slice, true_mask, pred_mask, slice_idx=1):
    
    ct_slice = np.squeeze(ct_slice)
    true_mask = np.squeeze(true_mask)
    pred_mask = np.squeeze(pred_mask)
    
    ct_crop, true_crop, pred_crop = crop_to_body(ct_slice, true_mask, pred_mask)

    cmap_masks = mcolors.ListedColormap(['none', '#1f77b4', '#2ca02c']) # Sfondo, Azzurro, Verde
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 6), dpi=300)
    fig.patch.set_facecolor('white') 
    
    # 1. CT Originale
    axes[0].imshow(ct_crop, cmap='gray')
    axes[0].set_title('CT Scan Originale', fontsize=14, pad=10)
    axes[0].axis('off')

    # 2. Ground Truth (Verità)
    axes[1].imshow(ct_crop, cmap='gray')
    mask_true_masked = np.ma.masked_where(true_crop == 0, true_crop)
    # AGGIUNTO vmin=0, vmax=2 PER BLOCCARE I COLORI!
    axes[1].imshow(mask_true_masked, cmap=cmap_masks, alpha=0.6, vmin=0, vmax=2)
    axes[1].set_title('Ground Truth', fontsize=14, pad=10)
    axes[1].axis('off')

    # 3. Predizione (AI)
    axes[2].imshow(ct_crop, cmap='gray')
    mask_pred_masked = np.ma.masked_where(pred_crop == 0, pred_crop)
    # AGGIUNTO vmin=0, vmax=2 PER BLOCCARE I COLORI!
    axes[2].imshow(mask_pred_masked, cmap=cmap_masks, alpha=0.6, vmin=0, vmax=2)
    axes[2].set_title(' U-Net Model Prediction', fontsize=14, pad=10)
    axes[2].axis('off')

    legend_elements = [
        Patch(facecolor='#1f77b4', alpha=0.6, label='Addominali (Classe 1)'),
        Patch(facecolor='#2ca02c', alpha=0.6, label='Psoas (Classe 2)')
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=2, bbox_to_anchor=(0.5, 0.05), fontsize=12, frameon=False)

    plt.tight_layout(rect=[0, 0.12, 1, 1]) 
    plt.savefig(f"thesis_Step2_Segmentation_Slice{slice_idx}.png", bbox_inches='tight', facecolor='white')
    plt.show()





## 8. Standard Case Validations

To demonstrate the model's standard performance, we extract multiple random batches from the validation generator. By dynamically scanning the batch for slices containing actual target structures (e.g., Abdominal muscles), we ensure our qualitative examples accurately represent the model's primary task.

In [ ]:
X_val_batch, y_val_batch = val_generator[0]

preds_batch = model.predict(X_val_batch, verbose=0)
pred_classes_batch = np.argmax(preds_batch, axis=-1)

# Cerchiamo dinamicamente un'immagine nel batch che abbia sicuramente gli addominali
for i in range(X_val_batch.shape[0]):
    if np.any(y_val_batch[i] == 1): # Controlla se c'è almeno 1 pixel di classe 1 (Addominali)
        idx_scelto = 0
 
        break

plot_step2_segmentation(
    ct_slice=X_val_batch[idx_scelto], 
    true_mask=y_val_batch[idx_scelto], 
    pred_mask=pred_classes_batch[idx_scelto], 
    slice_idx=idx_scelto 
)

In [ ]:
X_val_batch, y_val_batch = val_generator[1]

preds_batch = model.predict(X_val_batch, verbose=0)
pred_classes_batch = np.argmax(preds_batch, axis=-1)

# Cerchiamo dinamicamente un'immagine nel batch che abbia sicuramente gli addominali
idx_scelto = 2
for i in range(X_val_batch.shape[0]):
    if np.any(y_val_batch[i] == 1): # Controlla se c'è almeno 1 pixel di classe 1 (Addominali)
        idx_scelto = 0
        break

plot_step2_segmentation(
    ct_slice=X_val_batch[idx_scelto], 
    true_mask=y_val_batch[idx_scelto], 
    pred_mask=pred_classes_batch[idx_scelto], 
    slice_idx=idx_scelto 
)

In [ ]:
X_val_batch, y_val_batch = val_generator[2]

preds_batch = model.predict(X_val_batch, verbose=0)
pred_classes_batch = np.argmax(preds_batch, axis=-1)

# Cerchiamo dinamicamente un'immagine nel batch che abbia sicuramente gli addominali
idx_scelto = 2
for i in range(X_val_batch.shape[0]):
    if np.any(y_val_batch[i] == 1): # Controlla se c'è almeno 1 pixel di classe 1 (Addominali)
        idx_scelto = 0
        break

plot_step2_segmentation(
    ct_slice=X_val_batch[idx_scelto], 
    true_mask=y_val_batch[idx_scelto], 
    pred_mask=pred_classes_batch[idx_scelto], 
    slice_idx=idx_scelto 
)

## 9. Clinical Auditing: The Worst-Case Scenario

A robust clinical model assessment must actively search for its own failures. In this final step, we iterate through the entire validation set to isolate the single prediction generating the highest Maximum Hausdorff Distance (in millimeters). 

Plotting this specific "Worst-Case Scenario" is highly valuable for error analysis. It reveals whether the mathematical error stems from a true morphological failure in the abdominal wall, or from an isolated false-positive artifact (e.g., misclassifying arm tissue as abdominal muscle), which can be easily resolved with standard post-processing techniques.

In [ ]:
# --- 1. RICERCA DELLA PEGGIORE PREDIZIONE (WORST CASE SCENARIO IN MILLIMETRI) ---

worst_hd_mm = 0.0
worst_batch_idx = -1
worst_slice_idx = -1

TARGET_SIZE = 256
batch_size = val_generator.batch_size

print("Ricerca dell'errore massimo (Max HD) in millimetri...")

# Iteriamo su tutto il validation set per trovare l'errore massimo
for i in range(len(val_generator)):
    X_batch, y_batch = val_generator[i]
    
    preds = model.predict(X_batch, verbose=0)
    pred_classes = np.argmax(preds, axis=-1)
    true_classes = np.squeeze(y_batch, axis=-1) 

    # Estraiamo i percorsi reali dei file NIfTI per questo batch (serve per i millimetri)
    batch_paths = val_img[i * batch_size : (i + 1) * batch_size]

    for j in range(X_batch.shape[0]):
        true_mask = true_classes[j]
        pred_mask = pred_classes[j]
        
        # Estrazione spaziatura in millimetri
        nii_file = nib.load(batch_paths[j])
        original_zooms = nii_file.header.get_zooms()
        original_shape = nii_file.shape
        
        factor_x = original_shape[0] / TARGET_SIZE
        factor_y = original_shape[1] / TARGET_SIZE
        
        spacing_x_mm = original_zooms[0] * factor_x
        spacing_y_mm = original_zooms[1] * factor_y

        # Isoliamo gli addominali (Classe 1)
        true_abs = (true_mask == 1)
        pred_abs = (pred_mask == 1)
        
        # Calcoliamo il Max HD in MILLIMETRI
        hd_abs_mm = calculate_HD_single_class_mm(true_abs, pred_abs, spacing_y_mm, spacing_x_mm)
        
        # Se troviamo un errore in mm più alto di quello registrato finora, lo salviamo
        if not np.isnan(hd_abs_mm) and hd_abs_mm > worst_hd_mm:
            worst_hd_mm = hd_abs_mm
            worst_batch_idx = i
            worst_slice_idx = j

print(f"Batch: {worst_batch_idx} | Slice nel batch: {worst_slice_idx}")
print(f"Errore Max HD: {worst_hd_mm:.2f} mm")

# --- 2. ESTRAZIONE E PLOT DEL CASO PEGGIORE ---
# Ora che sappiamo dove si trova l'errore, recuperiamo quel batch specifico
X_worst_batch, y_worst_batch = val_generator[worst_batch_idx]
preds_worst_batch = model.predict(X_worst_batch, verbose=0)
pred_classes_worst_batch = np.argmax(preds_worst_batch, axis=-1)

# Passiamo la fetta specifica alla tua funzione di plotting invariata
plot_step2_segmentation(
    ct_slice=X_worst_batch[worst_slice_idx], 
    true_mask=y_worst_batch[worst_slice_idx], 
    pred_mask=pred_classes_worst_batch[worst_slice_idx], 
    # Usiamo lo slice_idx come etichetta per il file salvato indicando "mm"
    slice_idx=f"WorstCase_Batch{worst_batch_idx}_Slice{worst_slice_idx}_Err{worst_hd_mm:.1f}mm" 
)

## Pipeline Architecture: Model Training and Evaluation

This flowchart illustrates the end-to-end process from data ingestion into the neural network to the final clinical and visual evaluation.

```text

                 PHASE 1: ON-THE-FLY GENERATION & COMPILATION

[ Patient-Isolated Datasets (Train / Val) ]
       │
       ├──> L3MuscleDataGenerator (Batching & HU Windowing)
       │
       ▼
[ Custom U-Net Architecture ]
       │
       ├──> Encoder-Decoder Depth: 16 -> 32 -> 64 -> 128 -> 256
       ├──> Optimizer: Adam (lr=0.001)
       └──> Objective: Combined Loss (Sparse Categorical Crossentropy + Dice Loss)



                 PHASE 2: TRAINING & CHECKPOINTING

[ Model Fit (50 Epochs) ]
       │
       ├──> Monitor: Validation Loss
       └──> Callback: ModelCheckpoint (Save Best Only)
       │
       ▼
[ Saved Weights: unet_sarcopenia_best.keras ]



                 PHASE 3: PHYSICAL QUANTITATIVE VALIDATION

[ Inference on Validation Set ]
       │
       ├──> 1. Extract physical voxel spacing (mm) from native NIfTI headers
       ├──> 2. Convert boundary coordinates from pixels to physical space
       └──> 3. Compute Metrics:
                - Balanced Average Hausdorff Distance (bAHD in mm)
                - Maximum Hausdorff Distance (Max HD in mm)
       │
       ▼
[ Statistical Distribution ]  ==>  [ Boxplot Generation (Seaborn) ]



                 PHASE 4: QUALITATIVE CLINICAL ANALYTICS

[ Segmentation Mask Output (Classes 0, 1, 2) ]
       │
       ├──> 1. Dynamic Cropping (Isolating the anatomical body wall)
       ├──> 2. Standard Case Visualization (Overlay of original, true, predicted)
       └──> 3. Worst-Case Scenario Audit:
                - Programmatically scan for the highest Max HD error.
                - Extract and plot the exact slice to isolate false-positives.
       │
       ▼
[ Final Validated Medical Software Pipeline ]